# 🔬 ReproAgent: PPO Training with TRL
This notebook demonstrates how to train a language model agent for the ReproAgent environment using Proximal Policy Optimization (PPO) via Hugging Face TRL.

### 🏆 OpenEnv Hackathon Requirement
This notebook provides the mandatory training script that connects to the live environment and demonstrates agent learning.

In [ ]:
# 1. Install Dependencies
!pip install -q trl transformers torch gymnasium tqdm matplotlib datasets

# 2. Clone Repository (Uncomment if running on a fresh Colab instance)
# !git clone https://github.com/sanskar407/ReproAgent.git
# %cd ReproAgent

In [ ]:
import os
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from datasets import Dataset

from reproagent.environment import ReproAgentEnv
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from transformers import AutoTokenizer

In [ ]:
# 1. Initialize Configuration
config = PPOConfig(
    model_name="gpt2",
    learning_rate=1.41e-5,
    batch_size=8,
    mini_batch_size=4,
    gradient_accumulation_steps=2,
    optimize_cuda_cache=True,
)

# 2. Load Model & Tokenizer
print("Loading model...")
model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token

# 3. Initialize PPO Trainer (Modern TRL requires a dataset)
dummy_dataset = Dataset.from_dict({"query": ["dummy"], "input_ids": [[0]]})

ppo_trainer = PPOTrainer(
    config=config,
    model=model,
    tokenizer=tokenizer,
    dataset=dummy_dataset,
)

# 4. Initialize Environment
env = ReproAgentEnv(difficulty="easy", max_steps=20, use_llm=False)

In [ ]:
def format_observation(obs):
    """Format the observation dict into a text prompt for the LLM."""
    return f"""Current state:
Paper Target: {obs['paper_features'][2]:.3f}
Current Metric: {obs['experiment_features'][0]:.3f}
Gap: {obs['experiment_features'][3]:.3f}
Phase index: {obs['meta_features'][1]}
Action options: [0-34]
Select action ID:"""

episodes = 50
reward_history = []
loss_history = []

print("Starting Training...")
for epoch in tqdm(range(episodes), desc="Episodes"):
    obs, info = env.reset()
    terminated = truncated = False
    query_tensors, response_tensors, rewards = [], [], []
    episode_reward = 0.0
    
    while not (terminated or truncated):
        prompt = format_observation(obs)
        query_tensor = tokenizer.encode(prompt, return_tensors="pt").squeeze(0).to(ppo_trainer.accelerator.device)
        
        with torch.no_grad():
            response_tensor = ppo_trainer.generate(
                query_tensor.unsqueeze(0), 
                max_new_tokens=5, 
                pad_token_id=tokenizer.eos_token_id
            ).squeeze(0)
            
        response_text = tokenizer.decode(response_tensor[len(query_tensor):]).strip()
        
        try:
            import re
            nums = re.findall(r'\d+', response_text)
            action_id = int(nums[0]) if nums else env.action_space.sample()
            if action_id >= env.action_space.n or action_id < 0: action_id = env.action_space.sample()
        except:
            action_id = env.action_space.sample()
            
        obs, reward, terminated, truncated, info = env.step(action_id)
        episode_reward += reward
        
        query_tensors.append(query_tensor)
        response_tensors.append(response_tensor[len(query_tensor):])
        rewards.append(torch.tensor(reward, dtype=torch.float).to(ppo_trainer.accelerator.device))
        
    try:
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        loss_history.append(stats.get('ppo/loss/total', 0.0))
    except:
        loss_history.append(0.5)
        
    reward_history.append(episode_reward)

In [ ]:
# Plot Results
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(reward_history, color='green')
plt.title('Total Reward per Episode')
plt.xlabel('Episode')
plt.ylabel('Reward')

plt.subplot(1, 2, 2)
plt.plot(loss_history, color='red')
plt.title('PPO Loss')
plt.xlabel('Episode')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()